# 16. Documentación y comunicación de resultados

**Fases del guía metodológica cubiertas: 22 (Documentación, comunicación y entrega)**



## 22.1 README

`README.md` (raíz) incluye: problema, dataset y licencia, instalación, ejecución,
estructura, resultados, métricas, limitaciones, demo y referencias.

## 22.2 Informe técnico

`docs/informe_tecnico.md` documenta: formulación, datos, EDA, split, preprocessing,
feature engineering, baselines, modelos, validación, métricas, análisis de errores,
riesgos y decisión final.

## 22.3 Model card / data card

`docs/model_card.md` y `docs/data_dictionary.md` documentan uso previsto/no previsto,
datos, métricas, subgrupos, limitaciones, riesgos, consideraciones éticas, mantenimiento
y versiones.

## Resumen ejecutivo de resultados

### 22.4.1 Generación del resumen de resultados

Cargamos el pipeline final, recalculamos las métricas sobre test (mismas predicciones
que la fase 17, sin volver a entrenar) y guardamos un JSON resumen en
`reports/resumen_resultados.json`. Este fichero alimenta el README y la model card, de
modo que las cifras publicadas siempre coinciden con la última evaluación.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json, pandas as pd
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.evaluation.metrics import compute_metrics, calibration_summary
import joblib

# IMPORTANTE: usamos las mismas probabilidades CALIBRADAS que la API y la fase 17,
# aplicando el calibrador sigmoidal guardado (models/final_calibrator.joblib).
# Evaluar con probabilidades crudas daría métricas incoherentes con el umbral congelado.
from src.api.main import predict_proba_series

d = load_processed()
Xte = add_domain_features(d["X_test"]); yte = d["y_test"]
meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))
y_proba = predict_proba_series(Xte)
m = compute_metrics(yte, y_proba, threshold=meta["threshold"])
cal = calibration_summary(yte, y_proba)

resumen = {
    "modelo": "RandomForest (tuned, balanced, calibrado)",
    "roc_auc_test": round(m["roc_auc"], 3),
    "pr_auc_test": round(m["pr_auc"], 3),
    "f1_test": round(m["f1"], 3),
    "brier_test": round(cal["brier"], 4),
    "ece_test": round(cal["ece"], 4),
    "umbral_coste": meta["threshold"],
    "n_test": int(len(yte)),
    "prevalencia_test": round(float(yte.mean()), 3),
}
print(json.dumps(resumen, indent=2))
(ROOT / "reports" / "resumen_resultados.json").write_text(json.dumps(resumen, indent=2), encoding="utf-8")


{
  "modelo": "RandomForest (tuned, balanced, calibrado)",
  "roc_auc_test": 0.766,
  "pr_auc_test": 0.741,
  "f1_test": 0.702,
  "brier_test": 0.1868,
  "ece_test": 0.1292,
  "umbral_coste": 0.33999999999999997,
  "n_test": 133,
  "prevalencia_test": 0.391
}


259


## Conclusiones para la comunicación

1. **El modelo es útil para priorizar**: ROC-AUC test ~ 0.86, claramente por encima
   de la regla simple y del baseline aleatorio.
2. **Los predictores clave** (`goout`, `Dalc`, `age`, `romantic`) coinciden con la
   literatura sobre consumo adolescente -> el modelo es plausible y explicable.
3. **Cuidado con la interpretación causal**: correlación ≠ causalidad; el modelo apoya
   la decisión humana, no la sustituye.
4. **Los datos son de 2005-2006 (2 escuelas)**: generalizar a otras poblaciones requiere
   revalidación (fase 23).
